PCA
Decision Trees/Forest
Neural Network

Preprocessing (drop columns and define predictors/outcome)

In [ ]:
# drop game title and release year columns
game_df = game_df.drop(columns=['Game Title', 'Release Year', 'Developer', 'Publisher', 'User Review Text'])

# encode some of the categorical variables into numerical
dummy_cols = ['Game Mode', 'Multiplayer', 'Age Group Targeted', 'Requires Special Device', 'Platform', 'Genre']
label_cols = ['Graphics Quality','Soundtrack Quality','Story Quality']

le = LabelEncoder()
for col in label_cols:
  game_df[col] = le.fit_transform(game_df[col])

game_df = pd.get_dummies(game_df, columns=dummy_cols, drop_first=True)
print(game_df.info())
game_df.head()

In [ ]:
X = game_df.drop(columns = ['Price'])
y = game_df['Price']
# Split data between training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=67)

In [ ]:
# 1. Calculate the average price from the training set
price_mean = y_train.mean()

# 2. Create a list of predictions where every prediction is just that average
# We make it the same length as y_test
baseline_pred = [price_mean] * len(y_test)

# 3. Evaluate
rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
r2 = r2_score(y_test, baseline_pred)

print(f"Average Price (Baseline Prediction): {price_mean:.2f}")
print("RMSE=", rmse)
print("R2=", r2)

Full tree

In [ ]:
full_clf = DecisionTreeRegressor(random_state=67)
full_clf.fit(X_train, y_train)


In [ ]:
plt.figure(figsize=(35,10))
tree.plot_tree(full_clf, feature_names=X.columns, filled=True, max_depth=4, fontsize=10)

In [ ]:
# Evaluate the accuracy of the tree
y_pred_reg = full_clf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test,y_pred_reg))
r2 = r2_score(y_test,y_pred_reg)
print("RMSE=", rmse)
print("R2=", r2)

In [ ]:
# We create lists of all the values we want to search over
# This is called 'creating a grid'

# Then, use the GridSearchCV function which combines gridsearch with
# K-fold cross validation (cv parameter)
param_grid = {
    'max_depth': [10, 20, 30, 40],
    'min_samples_leaf': [20, 50, 100, 200],
    'min_impurity_decrease': [0, 0.0005, 0.001, 0.005, 0.01],
}

gridSearch = GridSearchCV(DecisionTreeRegressor(), param_grid, cv=5, scoring = 'r2')
gridSearch.fit(X_train, y_train)

print('Best R2 score: ', gridSearch.best_score_)
print('Parameter values to achieve best accuracy: ', gridSearch.best_params_)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth': [10, 20, 30, 40],
    'min_samples_leaf': [20, 50, 100, 200],
    'min_impurity_decrease': [0, 0.0005, 0.001, 0.005, 0.01],
}

randSearch = RandomizedSearchCV(DecisionTreeRegressor(random_state=67),
                                param_distributions=param_dist,
                                n_iter=10,
                                cv=5,
                                scoring='r2',
                                random_state=67)
randSearch.fit(X_train, y_train)

print('Best R2 score: ', randSearch.best_score_)
print('Parameter values to achieve best score: \n', randSearch.best_params_)

In [ ]:
final_clf = DecisionTreeRegressor(min_samples_leaf = 50, min_impurity_decrease = 0, max_depth = 20, random_state=67)
final_clf.fit(X_train, y_train)

# Evaluate the accuracy of the tree
y_pred_reg = final_clf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test,y_pred_reg))
r2 = r2_score(y_test,y_pred_reg)
print(f"Decision Tree Root Mean Squared Error: {rmse}")
print(f"Decision Tree R^2 Score: {r2}")

In [ ]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=0)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Root Mean Squared Error: {mse_rf}")
print(f"Random Forest R^2 Score: {r2_rf}")

In [ ]:
# Extract importance values for each feature (column of X)
importances = rf.feature_importances_

# create a dataframe to store the values and their labels
df2 = pd.DataFrame({'feature': X_train.columns, 'importance': importances})

# sort dataframe by descending order, showing the most important feature top
df2 = df2.sort_values('importance', ascending = False)

# plot the importance of each feature
ax = df2.plot(kind='bar', x='feature')

plt.tight_layout()
plt.show()